# BOT-2 Fusion V4 —
Compares CNN-only, OpenCV + Random Forest, and fusion on identical splits. Uses capped class-balanced loss (default) without double-counting via oversampling. Saves score audits, plots, confusion matrices, metrics, predictions, checkpoints, and ZIP. Test metrics are final reporting only; tune on validation.


In [ ]:
import os, random, json, re, zipfile, warnings, copy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error, confusion_matrix,
                             cohen_kappa_score, classification_report)
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
ROOT=Path('/kaggle/input/datasets/huss69/shape-augmented-dataset/results/dataset_by_class')
OUT=Path('/kaggle/working/shape_comparison_results_v4'); OUT.mkdir(parents=True,exist_ok=True)
IMG_SIZE=384; EPOCHS=75; PATIENCE=15; BATCH=16; LR=1e-4; WORKERS=2; FREEZE_EPOCHS=5; ORDINAL_WEIGHT=0.35; LABEL_SMOOTH=0.0
PRETRAINED=True; RUN_CNN=True; RUN_RF=True; RUN_FUSION=True
MAX={'circle':4,'wave':4,'square':5,'triangle':5,'diamond':5,'star':5,'overlapped_circles':6,'overlapped_pencils':6}
if not ROOT.exists():
    found=list(Path('/kaggle/input').rglob('dataset_by_class')) if Path('/kaggle/input').exists() else []
    if not found: raise FileNotFoundError('Attach shape-augmented-dataset in Kaggle Add Input. Expected dataset_by_class/train,val,test')
    ROOT=next((p for p in found if (p/'train').exists()),found[0])
print('Dataset:',ROOT,'GPU:',DEVICE)
print('Saving to:',OUT)
# V3: validation-only checkpoint selection; do not tune on test data.
ACCURACY_WEIGHT=0.55; QWK_WEIGHT=0.45

# V4: use capped effective-number class weights; do not combine with oversampling by default.
BALANCE_MODE="loss"  # "loss", "sampler", or "none"; choose using validation only
BETA=0.99; MAX_CLASS_WEIGHT=3.0; BALANCED_CE_STRENGTH=0.65


Dataset: /kaggle/input/datasets/huss69/shape-augmented-dataset/results/dataset_by_class GPU: cuda
Saving to: /kaggle/working/shape_comparison_results_v4


In [ ]:
rows=[]
for split in ['train','val','test']:
    folder=ROOT/split
    if not folder.exists(): raise FileNotFoundError(folder)
    for shdir in sorted(folder.iterdir()):
        if not shdir.is_dir():continue
        for sd in sorted(shdir.iterdir()):
            m=re.fullmatch(r'score_(\d+)',sd.name)
            if not sd.is_dir() or not m:continue
            for p in sorted(sd.iterdir()):
                if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}:
                    rows.append(dict(path=str(p),split=split,shape=shdir.name,score=int(m.group(1)),augmented=('_aug' in p.stem.lower())))
df=pd.DataFrame(rows)
if df.empty:raise RuntimeError('No images found')
for sh,g in df.groupby('shape'):
    if sh not in MAX:MAX[sh]=int(g.score.max());print('Unrecognized shape:',sh,'using max',MAX[sh])
    if (g.score>MAX[sh]).any():raise ValueError('Score exceeds permitted maximum for '+sh)
shapes=sorted(df['shape'].unique()); sid={s:i for i,s in enumerate(shapes)}
df['sid']=df['shape'].map(sid); df['row_id']=np.arange(len(df))
print(pd.crosstab([df['shape'],df['score']],df['split']))
print('Augmented in validation/test:',df.loc[df.split!='train','augmented'].sum())
if df.loc[df.split!='train','augmented'].any():raise ValueError('Augmented images found in val/test: check dataset split')
# Heuristic leakage audit: augmented filenames must not have original counterparts in another split.
def parent_key(p):
    name=Path(p).stem
    return re.sub(r'(?i)_aug(?:[_-]?\d+.*)?$','',name)
df['source_key']=df.path.map(parent_key)
leak=df.groupby(['shape','source_key'])['split'].nunique()
if (leak>1).any():
    print('WARNING: possible cross-split original/augmentation leakage:',int((leak>1).sum()))
    print('Inspect source_key naming before trusting test scores.')
df.to_csv(OUT/'dataset_manifest.csv',index=False)
pd.crosstab([df['shape'],df['score']],df['split']).to_csv(OUT/'class_distribution.csv')
fig,ax=plt.subplots(figsize=(12,5));pd.crosstab(df['shape'],df['split']).plot.bar(ax=ax);ax.set_ylabel('Images');plt.tight_layout();fig.savefig(OUT/'dataset_distribution.png',dpi=180);plt.close(fig)
# Audit counts of independent original drawings, score-wise coverage and majority baselines.
audit=[]
for (shape,score),g in df.groupby(['shape','score']):
    audit.append(dict(shape=shape,score=int(score),split_counts=g.groupby('split').size().to_dict(),
                      train_images=int((g.split=='train').sum()),
                      train_originals=int(((g.split=='train')&(~g.augmented)).sum()),
                      train_unique_sources=int(g.loc[g.split=='train','source_key'].nunique())))
pd.DataFrame(audit).drop(columns='split_counts').to_csv(OUT/'score_audit.csv',index=False)
for sh in shapes:
    for split in ['val','test']:
        sub=df[(df['shape']==sh)&(df.split==split)]
        if len(sub):
            major=int(df[(df['shape']==sh)&(df.split=='train')].score.mode().iloc[0])
            print('Majority baseline:',sh,split,round(float((sub.score==major).mean()),4))
# Hard fail on clear original/augmented cross-split filename collisions.
if (leak>1).any():
    raise ValueError('Possible cross-split leakage detected. Resolve source_key collisions before training.')


split                     test  train  val
shape              score                  
circle             0         1     23    1
                   2         0      3    0
                   3        17    317   17
                   4        69    319   69
diamond            0         1     20    1
                   3         1     10    0
                   4        10    334   10
                   5        72    334   72
overlapped_circles 0         1     10    1
                   2         0     10    0
                   4         1     70    1
                   5        20    270   20
                   6        58    270   58
overlapped_pencils 0         2     80    2
                   2         1     10    0
                   3         1     60    1
                   4         8    158    8
                   5        21    158   21
                   6        34    158   34
square             0         0     10    0
                   4         7    310    7
           

In [ ]:
def ink_map(bgr):
    dark=255-bgr.min(axis=2)
    k=max(15,(min(dark.shape)//15)|1)
    bg=cv2.morphologyEx(dark,cv2.MORPH_OPEN,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k)))
    ink=cv2.subtract(dark,bg).astype(np.float32)
    hi=np.percentile(ink,99.7)
    return np.zeros_like(ink) if hi<1 else np.clip(ink/hi,0,1)
def pad_resize(a,size):
    h,w=a.shape; s=max(h,w); canvas=np.zeros((s,s),dtype=a.dtype)
    canvas[(s-h)//2:(s-h)//2+h,(s-w)//2:(s-w)//2+w]=a
    return cv2.resize(canvas,(size,size),interpolation=cv2.INTER_AREA if s>size else cv2.INTER_LINEAR)
def geo_features(im):
    # Geometry derived from an un-cropped, padded ink map. All features numerical, no expert facet labels assumed.
    mask=(im>max(.15,float(np.quantile(im,.90))*.4)).astype(np.uint8)
    H,W=mask.shape; ys,xs=np.where(mask>0)
    if len(xs)<8:return np.zeros(32,dtype=np.float32)
    x0,x1=xs.min(),xs.max();y0,y1=ys.min(),ys.max();bw=x1-x0+1;bh=y1-y0+1
    cnts,_=cv2.findContours(mask,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    cnts=sorted(cnts,key=cv2.contourArea,reverse=True)
    cnt=cnts[0] if cnts else None
    area=float(cv2.contourArea(cnt)) if cnt is not None else 0.
    peri=float(cv2.arcLength(cnt,True)) if cnt is not None else 0.
    hull=cv2.convexHull(cnt) if cnt is not None else None
    ha=float(cv2.contourArea(hull)) if hull is not None else 0.
    approx=cv2.approxPolyDP(cnt,.025*peri,True) if peri>0 else []
    num_corners=len(approx)
    edges=np.linalg.norm(np.diff(np.concatenate([np.asarray(approx).reshape(-1,2),np.asarray(approx).reshape(-1,2)[:1]],axis=0),axis=0),axis=1) if num_corners>1 else np.array([0.])
    try:
        from skimage.morphology import skeletonize
        sk=skeletonize(mask.astype(bool)).astype(np.uint8)*255
    except ImportError:
        sk=(mask*255).astype(np.uint8)
    sk=(sk>0).astype(np.uint8)
    neigh=cv2.filter2D(sk.astype(np.float32),-1,np.ones((3,3),np.float32))-sk
    endpoints=float(((sk>0)&(neigh==1)).sum());junctions=float(((sk>0)&(neigh>=3)).sum())
    moments=cv2.moments(mask)
    if moments['m00']>0:
        cov=np.cov(np.stack([xs,ys]));evals,evecs=np.linalg.eigh(cov);angle=float(np.arctan2(evecs[1,-1],evecs[0,-1])/np.pi)
        elong=float(evals[-1]/max(evals[0],1e-6))
    else:angle=0.;elong=0.
    roi=mask[y0:y1+1,x0:x1+1]
    symx=1-float(np.mean(np.abs(roi.astype(float)-np.fliplr(roi).astype(float))))
    symy=1-float(np.mean(np.abs(roi.astype(float)-np.flipud(roi).astype(float))))
    ncc,lab,stats,_=cv2.connectedComponentsWithStats(mask,8)
    significant=int(np.sum(stats[1:,cv2.CC_STAT_AREA]>=max(5,.01*len(xs))))
    f=[len(xs)/(H*W),bw/W,bh/H,bw/max(bh,1),max(bw,bh)/max(min(bw,bh),1),
       (bw*bh)/(H*W),x0/W,y0/H,(W-x1)/W,(H-y1)/H,
       area/(H*W),peri/(H+W),4*np.pi*area/(peri**2+1e-6),area/(ha+1e-6),
       num_corners/20.,float(np.max(edges)/(np.min(edges[edges>0])+1e-6)) if (edges>0).any() else 0.,
       endpoints/max(len(xs),1),junctions/max(len(xs),1),significant/10.,angle,
       min(elong,100)/100.,symx,symy,len(cnts)/30.]
    # Additional size-preserving descriptors: quantiles, directional ink density and center-of-mass.
    f.extend([float(np.quantile(xs/W,q)) for q in (.1,.5,.9)])
    f.extend([float(np.quantile(ys/H,q)) for q in (.1,.5,.9)])
    f.extend([float(np.mean(mask[:H//2])),float(np.mean(mask[:, :W//2]))])
    return np.nan_to_num(np.asarray(f,np.float32),nan=0,posinf=0,neginf=0)
def process(path):
    bgr=cv2.imdecode(np.fromfile(path,np.uint8),cv2.IMREAD_COLOR)
    if bgr is None:raise ValueError('Unreadable image '+path)
    ink=pad_resize(ink_map(bgr),IMG_SIZE)
    return (np.clip(ink,0,1)*255).astype(np.uint8),geo_features(ink)
print('Preprocessing',len(df),'images; no augmentation performed...')
with ThreadPoolExecutor(max_workers=min(4,os.cpu_count() or 2)) as pool:
    processed=list(pool.map(process,df.path.tolist()))
X=np.stack([v[0] for v in processed]);G=np.stack([v[1] for v in processed]);del processed
scaler=StandardScaler();tr=(df.split=='train').to_numpy();G=scaler.fit_transform(G[tr]) if False else G
# Fit ONLY training data; clip extreme standardized values.
scaler.fit(G[tr]);G=np.clip(scaler.transform(G),-10,10).astype(np.float32)
import joblib
joblib.dump(scaler,OUT/'geometry_scaler.joblib')
np.save(OUT/'geometry_feature_summary.npy',np.stack([G.mean(0),G.std(0)]))
print('Image tensor:',X.shape,'Geometry:',G.shape)

Preprocessing 6566 images; no augmentation performed...
Image tensor: (6566, 384, 384) Geometry: (6566, 32)


In [ ]:
def score_metrics(y,p,maximum):
    y=np.asarray(y,int);p=np.asarray(p,int)
    q=cohen_kappa_score(y,p,labels=list(range(maximum+1)),weights='quadratic') if len(y)>1 else np.nan
    return dict(n=len(y),accuracy=accuracy_score(y,p),macro_f1=f1_score(y,p,labels=list(range(maximum+1)),average='macro',zero_division=0),
                weighted_f1=f1_score(y,p,average='weighted',zero_division=0),mae=mean_absolute_error(y,p),
                rmse=float(np.sqrt(np.mean((y-p)**2))),within_1=float(np.mean(abs(y-p)<=1)),qwk=q)
def report_model(name,split,pred):
    sub=df[df.split==split].copy();sub['pred']=np.asarray(pred,int)
    sub['abs_error']=(sub.score-sub.pred).abs();sub['model']=name
    sub.to_csv(OUT/f'{name}_{split}_predictions.csv',index=False)
    rows=[]
    for sh,g in sub.groupby('shape'):
        rows.append(dict(model=name,split=split,shape=sh,**score_metrics(g.score,g.pred,MAX[sh])))
    allmetrics=score_metrics(sub.score,sub.pred,max(MAX.values()))
    allmetrics['qwk']=float(np.nanmean([r['qwk'] for r in rows])) # macro shape QWK, not pooled kappa
    rows.append(dict(model=name,split=split,shape='ALL (macro-shape QWK)',**allmetrics))
    tab=pd.DataFrame(rows);tab.to_csv(OUT/f'{name}_{split}_metrics.csv',index=False)
    for sh,g in sub.groupby('shape'):
        labels=list(range(MAX[sh]+1));cm=confusion_matrix(g.score,g.pred,labels=labels)
        pd.DataFrame(cm,index=labels,columns=labels).to_csv(OUT/f'{name}_{split}_cm_{sh}.csv')
        fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm,cmap='Blues')
        for i in range(len(labels)):
            for j in range(len(labels)):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8)
        ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xlabel='Predicted',ylabel='Expert',title=f'{name} | {sh} | {split}')
        fig.tight_layout();fig.savefig(OUT/f'{name}_{split}_cm_{sh}.png',dpi=150);plt.close(fig)
        (OUT/f'{name}_{split}_classification_{sh}.txt').write_text(classification_report(g.score,g.pred,labels=labels,zero_division=0))
    return tab
class ImageDataset(Dataset):
    def __init__(self,split):self.idx=np.flatnonzero((df.split==split).to_numpy())
    def __len__(self):return len(self.idx)
    def __getitem__(self,i):
        k=self.idx[i];a=torch.from_numpy(np.repeat(X[k:k+1],3,axis=0).astype(np.float32)/255.)
        # Pretrained RGB weights; grayscale ink map repeated across 3 channels.
        return a,torch.from_numpy(G[k]),int(df.score.iloc[k]),int(df.sid.iloc[k])
# Compute class weights using TRAIN ONLY. Never use validation/test frequencies.
from torch.utils.data import WeightedRandomSampler
train_df=df[df.split=='train'].copy()
counts=train_df.groupby(['shape','score']).size()
class_weights={}
for sh in shapes:
    values=[]
    for score in range(MAX[sh]+1):
        count=int(counts.get((sh,score),0))
        raw=(1-BETA)/(1-BETA**count) if count else 0.0
        values.append(raw)
    nonzero=np.array([v for v in values if v>0])
    mean=float(nonzero.mean()) if len(nonzero) else 1.0
    for score,v in enumerate(values):
        class_weights[(sh,score)]=float(np.clip((v/mean)**BALANCED_CE_STRENGTH,0.25,MAX_CLASS_WEIGHT)) if v else 0.0
pd.DataFrame([dict(shape=sh,score=sc,train_count=int(counts.get((sh,sc),0)),weight=class_weights[(sh,sc)])
              for sh in shapes for sc in range(MAX[sh]+1)]).to_csv(OUT/'training_class_weights.csv',index=False)
WEIGHT_MATRIX=torch.tensor([[class_weights.get((sh,sc),0.) for sc in range(7)] for sh in shapes],dtype=torch.float32,device=DEVICE)
sampler=None
if BALANCE_MODE=='sampler':
    weights=np.array([1/np.sqrt(counts.loc[(r.shape,r.score)]) for r in train_df.itertuples()],dtype=np.float64)
    weights=np.clip(weights,0,np.quantile(weights,.95))
    sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),num_samples=len(weights),replacement=True)
loaders={s:DataLoader(ImageDataset(s),batch_size=BATCH,shuffle=(s=='train' and sampler is None),sampler=(sampler if s=='train' else None),num_workers=WORKERS,pin_memory=(DEVICE=='cuda')) for s in ['train','val','test']}
class Net(nn.Module):
    def __init__(self,fusion):
        super().__init__();self.fusion=fusion
        try:
            weights=torchvision.models.ResNet18_Weights.DEFAULT if PRETRAINED else None
            self.backbone=torchvision.models.resnet18(weights=weights)
        except Exception as e:
            warnings.warn('Pretrained weights unavailable: '+str(e)+'; falling back to random initialization')
            self.backbone=torchvision.models.resnet18(weights=None)
        self.backbone.fc=nn.Identity()
        if fusion:
            self.geo=nn.Sequential(nn.Linear(G.shape[1],64),nn.ReLU(),nn.LayerNorm(64),nn.Dropout(.15))
            dim=576
        else:dim=512
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(dim,128),nn.ReLU(),nn.Dropout(.25),nn.Linear(128,MAX[s]+1)) for s in shapes])
    def forward(self,x,g,s):
        f=self.backbone(x)
        if self.fusion:f=torch.cat([f,self.geo(g)],dim=1)
        logits=torch.full((len(s),7),-1e4,device=x.device)
        for i,head in enumerate(self.heads):
            ix=torch.where(s==i)[0]
            if ix.numel():logits[ix,:MAX[shapes[i]]+1]=head(f[ix])
        return logits
def ordinal_aware_loss(logits,y,s=None):
    """Cross-entropy plus cumulative ordinal BCE and smooth expected-score loss.
    P(Y>k) is derived from softmax class probabilities; targets are 1[Y>k].
    Only thresholds valid for each shape are included (masked logits are -1e4).
    """
    ce_per=F.cross_entropy(logits,y,label_smoothing=LABEL_SMOOTH,reduction='none')
    if s is not None and BALANCE_MODE=='loss':
        sample_weights=WEIGHT_MATRIX[s,y]
        ce=(ce_per*sample_weights).sum()/sample_weights.sum().clamp_min(1e-6)
    else:
        ce=ce_per.mean()
    probs=F.softmax(logits,dim=1)
    cum=torch.flip(torch.cumsum(torch.flip(probs,dims=[1]),dim=1),dims=[1])[:,1:]
    thresholds=torch.arange(logits.shape[1]-1,device=logits.device)
    valid=(logits[:,:-1]>-1e3) & (logits[:,1:]>-1e3)
    targets=(y[:,None]>thresholds[None,:]).float()
    ord_bce=F.binary_cross_entropy(cum.clamp(1e-5,1-1e-5),targets,reduction='none')
    ord_bce=(ord_bce*valid).sum()/valid.sum().clamp_min(1)
    expected=(probs*torch.arange(logits.shape[1],device=logits.device)[None,:]).sum(1)
    mae_surrogate=F.smooth_l1_loss(expected,y.float())
    return ce+ORDINAL_WEIGHT*ord_bce+0.15*mae_surrogate

def validation_objective(accuracy,qwk):
    # QWK can be undefined if a shape has no score variation.
    return ACCURACY_WEIGHT*accuracy+QWK_WEIGHT*((qwk+1)/2)

def infer(net,split):
    net.eval();pred=[];true=[];ids=[];losses=[]
    with torch.no_grad():
        for x,g,y,s in loaders[split]:
            x,g,y,s=x.to(DEVICE),g.to(DEVICE),y.to(DEVICE),s.to(DEVICE)
            logits=net(x,g,s);losses.append((ordinal_aware_loss(logits,y,s).item(),len(y)))
            pred.extend(logits.argmax(1).cpu().tolist());true.extend(y.cpu().tolist());ids.extend(s.cpu().tolist())
    return np.array(pred),np.array(true),np.array(ids),sum(l*n for l,n in losses)/sum(n for _,n in losses)
def fit_net(name,fusion):
    torch.manual_seed(SEED);net=Net(fusion).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    # Warm-up only scoring heads; geometry and backbone subsequently fine-tuned.
    for p in net.backbone.parameters():p.requires_grad=False
    best=-1e9;best_state=None;bad=0;history=[];best_epoch=None
    for epoch in range(1,EPOCHS+1):
        if epoch==FREEZE_EPOCHS+1:
            for p in net.backbone.parameters():p.requires_grad=True
            print(name,': backbone unfrozen')
        net.train();
        if epoch<=FREEZE_EPOCHS:net.backbone.eval()
        total=0;count=0;correct=0
        for x,g,y,s in loaders['train']:
            x,g,y,s=x.to(DEVICE),g.to(DEVICE),y.to(DEVICE),s.to(DEVICE)
            opt.zero_grad(set_to_none=True);logits=net(x,g,s)
            loss=ordinal_aware_loss(logits,y,s);loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),1.0);opt.step()
            total+=loss.item()*len(y);count+=len(y);correct+=(logits.argmax(1)==y).sum().item()
        scheduler.step()
        p,t,ids,vloss=infer(net,'val')
        qvals=[score_metrics(t[ids==i],p[ids==i],MAX[sh])['qwk'] for i,sh in enumerate(shapes) if (ids==i).sum()>1]
        vq=float(np.nanmean(qvals)) if np.isfinite(qvals).any() else float('nan')
        vacc=accuracy_score(t,p)
        objective=validation_objective(vacc,vq) if np.isfinite(vq) else vacc
        history.append(dict(epoch=epoch,train_loss=total/count,train_accuracy=correct/count,val_loss=vloss,val_accuracy=vacc,val_macro_qwk=vq,val_objective=objective,lr=opt.param_groups[0]['lr']))
        print(f'{name}: epoch {epoch:02d} train_loss={total/count:.4f} val_loss={vloss:.4f} val_acc={vacc:.4f} val_QWK={vq:.4f}')
        if np.isfinite(objective) and objective>best+1e-5:
            best=objective;best_epoch=epoch;best_state=copy.deepcopy({k:v.detach().cpu() for k,v in net.state_dict().items()});bad=0
        else:bad+=1
        if bad>=PATIENCE:print('Early stopping');break
    pd.DataFrame(history).to_csv(OUT/f'{name}_history.csv',index=False)
    if best_state is None:raise RuntimeError('No valid validation checkpoint for '+name)
    print(f'{name}: selected epoch {best_epoch} with validation objective {best:.4f}')
    net.load_state_dict(best_state);torch.save(dict(state=best_state,shapes=shapes,max_score=MAX,fusion=fusion,image_size=IMG_SIZE),OUT/f'{name}_best.pt')
    hist=pd.DataFrame(history)
    fig,axs=plt.subplots(1,3,figsize=(15,4))
    axs[0].plot(hist.epoch,hist.train_loss,label='Train');axs[0].plot(hist.epoch,hist.val_loss,label='Validation');axs[0].set_title('Loss');axs[0].legend()
    axs[1].plot(hist.epoch,hist.train_accuracy,label='Train');axs[1].plot(hist.epoch,hist.val_accuracy,label='Validation');axs[1].set_title('Accuracy');axs[1].legend()
    axs[2].plot(hist.epoch,hist.val_macro_qwk,label='Validation QWK');axs[2].plot(hist.epoch,hist.val_objective,label='Validation objective');axs[2].set_title('Validation selection');axs[2].legend()
    fig.tight_layout();fig.savefig(OUT/f'{name}_learning_curves.png',dpi=180);plt.close(fig)
    return net
print('Data loaders ready')

Data loaders ready


In [ ]:
# Three-way evaluation: CNN alone, geometry-only RF, and fusion.
# NOTE: Do not change architecture/hyperparameters based on test performance.
results=[]
if RUN_RF:
    rf_pred={}
    for i,sh in enumerate(shapes):
        a=(df['split'].eq('train') & df['shape'].eq(sh)).to_numpy()
        if not a.any():raise RuntimeError('No training samples for '+sh)
        clf=RandomForestClassifier(n_estimators=350,min_samples_leaf=2,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
        clf.fit(G[a],df.loc[a,'score'].values)
        joblib.dump(clf,OUT/f'rf_{sh}.joblib')
        for split in ['val','test']:
            ix=np.flatnonzero((df.split.eq(split)&df['shape'].eq(sh)).to_numpy())
            rf_pred.setdefault(split,{})
            for j,p in zip(ix,clf.predict(G[ix])):rf_pred[split][j]=int(p)
    for split in ['val','test']:
        ix=np.flatnonzero(df.split.eq(split).to_numpy())
        preds=[rf_pred[split][j] for j in ix]
        results.append(report_model('opencv_rf',split,preds))
if RUN_CNN:
    cnn=fit_net('cnn_resnet18',False)
    for split in ['val','test']:
        preds,_,_,_=infer(cnn,split)
        results.append(report_model('cnn_resnet18',split,preds))
    del cnn
    if DEVICE=='cuda':torch.cuda.empty_cache()
if RUN_FUSION:
    fused=fit_net('fusion_resnet18_opencv',True)
    for split in ['val','test']:
        preds,_,_,_=infer(fused,split)
        results.append(report_model('fusion_resnet18_opencv',split,preds))
    del fused
    if DEVICE=='cuda':torch.cuda.empty_cache()
comparison=pd.concat(results,ignore_index=True);comparison.to_csv(OUT/'all_model_metrics.csv',index=False)
summary=comparison[comparison['shape'].str.startswith('ALL')].copy()
summary.to_csv(OUT/'comparison_summary.csv',index=False)
print(summary[['model','split','accuracy','macro_f1','mae','within_1','qwk']].to_string(index=False))
# Comparison figures
for metric in ['accuracy','macro_f1','mae','within_1','qwk']:
    pivot=summary.pivot(index='model',columns='split',values=metric)
    fig,ax=plt.subplots(figsize=(9,4));pivot.plot.bar(ax=ax,rot=15);ax.set_title('Model comparison: '+metric);ax.set_ylabel(metric)
    fig.tight_layout();fig.savefig(OUT/f'comparison_{metric}.png',dpi=180);plt.close(fig)
for metric in ['accuracy','qwk','mae']:
    sub=comparison[(comparison.split=='test') & (~comparison['shape'].str.startswith('ALL'))]
    fig,ax=plt.subplots(figsize=(14,5));sub.pivot(index='shape',columns='model',values=metric).plot.bar(ax=ax)
    ax.set_title('Test performance by shape: '+metric);ax.set_ylabel(metric)
    fig.tight_layout();fig.savefig(OUT/f'per_shape_{metric}.png',dpi=180);plt.close(fig)
# Error distribution and per-score breakdown
for model in comparison.model.unique():
    pred=pd.read_csv(OUT/f'{model}_test_predictions.csv')
    fig,ax=plt.subplots(figsize=(7,4));pred.abs_error.value_counts().sort_index().plot.bar(ax=ax)
    ax.set(xlabel='Absolute score error',ylabel='Samples',title=f'{model}: test error distribution')
    fig.tight_layout();fig.savefig(OUT/f'{model}_test_error_hist.png',dpi=180);plt.close(fig)
    pred.groupby(['shape','score']).agg(n=('pred','size'),accuracy=('abs_error',lambda x:(x==0).mean()),mae=('abs_error','mean')).to_csv(OUT/f'{model}_per_score_test.csv')
# Metadata and zipped archive
(OUT/'run_config.json').write_text(json.dumps(dict(root=str(ROOT),image_size=IMG_SIZE,epochs=EPOCHS,patience=PATIENCE,batch=BATCH,lr=LR,seed=SEED,shapes=shapes,max_score=MAX,device=DEVICE,balance_mode=BALANCE_MODE,beta=BETA,max_class_weight=MAX_CLASS_WEIGHT),indent=2))
archive=Path('/kaggle/working/shape_comparison_results_v4.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUT.parent))
print('ALL SAVED:',OUT,'\nZIP:',archive)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 137MB/s]


cnn_resnet18: epoch 01 train_loss=1.4539 val_loss=0.9420 val_acc=0.7380 val_QWK=0.0368
cnn_resnet18: epoch 02 train_loss=1.2019 val_loss=0.8454 val_acc=0.7473 val_QWK=0.0665
cnn_resnet18: epoch 03 train_loss=1.0914 val_loss=0.7867 val_acc=0.7566 val_QWK=0.0567
cnn_resnet18: epoch 04 train_loss=1.0083 val_loss=0.7719 val_acc=0.7473 val_QWK=0.0969
cnn_resnet18: epoch 05 train_loss=0.9491 val_loss=0.7509 val_acc=0.7566 val_QWK=0.1359
cnn_resnet18 : backbone unfrozen
cnn_resnet18: epoch 06 train_loss=0.8349 val_loss=0.6930 val_acc=0.7860 val_QWK=0.1919
cnn_resnet18: epoch 07 train_loss=0.4371 val_loss=0.7903 val_acc=0.7612 val_QWK=0.1579
cnn_resnet18: epoch 08 train_loss=0.2912 val_loss=0.7296 val_acc=0.7767 val_QWK=0.3644
cnn_resnet18: epoch 09 train_loss=0.2071 val_loss=0.8407 val_acc=0.8000 val_QWK=0.3562
cnn_resnet18: epoch 10 train_loss=0.1420 val_loss=0.8965 val_acc=0.7938 val_QWK=0.3509
cnn_resnet18: epoch 11 train_loss=0.0846 val_loss=0.9748 val_acc=0.8047 val_QWK=0.3469
cnn_resnet